## Cell 1: Imports & Configuration

In [44]:
from __future__ import annotations

import json
import math
import os
import re
import sys
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any

import numpy as np
import requests
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

# -------- User-editable inputs --------
USER_QUERY = "Internal web app for the finance team with a relational database backend"

# Where to cache the raw ARG dump. Re-uses an existing file if present.
ARG_CACHE_PATH = Path("../../plugin/skills/azure-enterprise-infra-planner/scripts/arg_raw_output_all.json")

# Azure OpenAI settings
AOAI_BASE_URL    = os.environ.get("AZURE_OPENAI_BASE_URL", "https://workloads-assistant-aoai.openai.azure.com/openai/v1")
AOAI_DEPLOYMENT  = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-5-mini")
AOAI_API_VERSION = os.environ.get("AZURE_OPENAI_API_VERSION", "preview")

# BM25 hyperparameters
BM25_K1 = 1.5
BM25_B = 0.75

# Ranking parameters
TOP_K = 5
RELATIVE_CUTOFF = 0.3   # keep RGs with score >= 0.3 * top_score
MIN_AGG_N = 3           # don't emit a claim if matched set < 3 RGs

# Confidence tiers
CONF_STRONG  = 0.7
CONF_PARTIAL = 0.4

## Cell 2: Azure OpenAI Client

In [45]:
_credential = DefaultAzureCredential()
_token_provider = get_bearer_token_provider(_credential, "https://cognitiveservices.azure.com/.default")

aoai = OpenAI(
    base_url=AOAI_BASE_URL,
    api_key="placeholder",
    default_query={"api-version": AOAI_API_VERSION},
    default_headers={"Authorization": f"Bearer {_token_provider()}"},
)
print(f"Azure OpenAI client ready (deployment={AOAI_DEPLOYMENT}).")

Azure OpenAI client ready (deployment=gpt-5-mini).


## Cell 3: Cached ARG Fetch

In [46]:
ARG_URL = "https://management.azure.com/providers/Microsoft.ResourceGraph/resources?api-version=2022-10-01"

ARG_KQL = """
Resources
| where type !startswith "microsoft.portal/"
| where type !startswith "providers.test/"
| where isempty(managedBy)
| where not (tags contains "hidden-") and not (tags contains "link:")
| where resourceGroup !startswith "mc_"
    and resourceGroup !startswith "databricks-rg-"
    and resourceGroup !startswith "azurebackuprg_"
    and resourceGroup !startswith "defaultresourcegroup-"
    and resourceGroup != "networkwatcherrg"
| project id, name, type, kind, location, resourceGroup, subscriptionId, sku, identity, properties
""".strip()


def fetch_arg_all(cache_path: Path) -> list[dict]:
    """Fetch all resources via Azure Resource Graph, with caching."""
    if cache_path.exists():
        print(f"Loading cached ARG data from {cache_path} ...")
        with open(cache_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        # Handle both list format and dict-with-data format
        if isinstance(data, list):
            print(f"  Loaded {len(data)} resources from cache (list format).")
            return data
        if isinstance(data, dict) and "data" in data:
            rows = data["data"]
            print(f"  Loaded {len(rows)} resources from cache (dict format).")
            return rows
        if isinstance(data, dict) and "resources" in data:
            rows = data["resources"]
            print(f"  Loaded {len(rows)} resources from cache (resources key).")
            return rows
        raise ValueError(f"Unexpected cache format: {type(data)}")

    print("Fetching resources from Azure Resource Graph (may take a moment) ...")
    cred = DefaultAzureCredential()
    token = cred.get_token("https://management.azure.com/.default").token
    headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

    all_rows: list[dict] = []
    skip_token: str | None = None
    page = 0

    while True:
        page += 1
        body: dict[str, Any] = {"query": ARG_KQL, "options": {"resultFormat": "objectArray"}}
        if skip_token:
            body["options"]["$skipToken"] = skip_token

        resp = requests.post(ARG_URL, headers=headers, json=body, timeout=120)
        resp.raise_for_status()
        payload = resp.json()

        rows = payload.get("data", [])
        all_rows.extend(rows)
        print(f"  Page {page}: received {len(rows)} rows (total so far: {len(all_rows)})")

        skip_token = payload.get("$skipToken")
        if not skip_token:
            break

    # Cache to disk
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    with open(cache_path, "w", encoding="utf-8") as f:
        json.dump(all_rows, f)
    print(f"Cached {len(all_rows)} resources to {cache_path}.")
    return all_rows


raw_resources = fetch_arg_all(ARG_CACHE_PATH)
print(f"\nTotal raw resources: {len(raw_resources)}")

Fetching resources from Azure Resource Graph (may take a moment) ...
  Page 1: received 1000 rows (total so far: 1000)
  Page 2: received 1000 rows (total so far: 2000)
  Page 3: received 1000 rows (total so far: 3000)
  Page 4: received 1000 rows (total so far: 4000)
  Page 5: received 1000 rows (total so far: 5000)
  Page 6: received 1000 rows (total so far: 6000)
  Page 7: received 1000 rows (total so far: 7000)
  Page 8: received 1000 rows (total so far: 8000)
  Page 9: received 1000 rows (total so far: 9000)
  Page 10: received 1000 rows (total so far: 10000)
  Page 11: received 1000 rows (total so far: 11000)
  Page 12: received 1000 rows (total so far: 12000)
  Page 13: received 1000 rows (total so far: 13000)
  Page 14: received 1000 rows (total so far: 14000)
  Page 15: received 1000 rows (total so far: 15000)
  Page 16: received 904 rows (total so far: 15904)
Cached 15904 resources to plugin\skills\azure-enterprise-infra-planner\scripts\arg_raw_output_all.json.

Total raw res

## Cell 4: Filter Auto-Created Resource Groups

In [47]:
AUTO_RG_PATTERNS = [
    re.compile(r"^MC_", re.IGNORECASE),
    re.compile(r"^NetworkWatcherRG$", re.IGNORECASE),
    re.compile(r"^DefaultResourceGroup-", re.IGNORECASE),
    re.compile(r"^Databricks-rg-", re.IGNORECASE),
    re.compile(r"^AzureBackupRG_", re.IGNORECASE),
    re.compile(r"^cloud-shell-storage-", re.IGNORECASE),
    re.compile(r"^dynamicsdeployments$", re.IGNORECASE),
    re.compile(r"^microsoft-network$", re.IGNORECASE),
    re.compile(r"^LogAnalyticsDefaultResources$", re.IGNORECASE),
    re.compile(r"^synapseworkspace-managedrg-", re.IGNORECASE),
    re.compile(r"^managedResourceGroup-", re.IGNORECASE),
    re.compile(r"^AzureArcTest$", re.IGNORECASE),
]


def is_auto_rg(rg_name: str) -> bool:
    return any(p.search(rg_name) for p in AUTO_RG_PATTERNS)


# Group resources by (subscriptionId, resourceGroup)
rg_map: dict[tuple[str, str], list[dict]] = defaultdict(list)
for r in raw_resources:
    key = (r.get("subscriptionId", ""), r.get("resourceGroup", ""))
    rg_map[key].append(r)

total_rgs = len(rg_map)

# Drop auto-created RGs
rg_map = {k: v for k, v in rg_map.items() if not is_auto_rg(k[1])}

dropped = total_rgs - len(rg_map)
total_resources = sum(len(v) for v in rg_map.values())
print(f"Resource groups: {total_rgs} total -> {len(rg_map)} after filtering ({dropped} auto-created dropped)")
print(f"Resources after filtering: {total_resources}")

Resource groups: 1493 total -> 1482 after filtering (11 auto-created dropped)
Resources after filtering: 15886


## Cell 5: Whitelist Properties

In [48]:
def whitelist_resource(r: dict) -> dict:
    """Strip a resource dict to only the fields useful for analysis."""
    props = r.get("properties") if isinstance(r.get("properties"), dict) else {}
    sku = r.get("sku") if isinstance(r.get("sku"), dict) else {}
    identity = r.get("identity") if isinstance(r.get("identity"), dict) else {}

    def _safe_get(d, *path):
        cur = d
        for k in path:
            if isinstance(cur, dict):
                cur = cur.get(k)
            else:
                return None
        return cur

    def _network_refs(props):
        if not isinstance(props, dict):
            return {}
        refs = {}
        subnet_id = _safe_get(props, "subnet", "id") or _safe_get(props, "virtualNetworkSubnetId")
        if not subnet_id:
            subnet_id = _safe_get(props, "networkProfile", "virtualNetworkSubnetId")
        if subnet_id:
            refs["subnetId"] = subnet_id
        if props.get("privateEndpointConnections"):
            refs["hasPrivateEndpoints"] = True
        if _safe_get(props, "publicNetworkAccess"):
            refs["publicNetworkAccess"] = props["publicNetworkAccess"]
        return refs

    return {
        "name": r.get("name"),
        "type": r.get("type"),
        "kind": r.get("kind"),
        "location": r.get("location"),
        "resourceGroup": r.get("resourceGroup"),
        "subscriptionId": r.get("subscriptionId"),
        "sku": {k: sku.get(k) for k in ("name", "tier", "size", "family", "capacity") if sku.get(k) is not None} or None,
        "identityType": identity.get("type"),
        "tier": props.get("tier") or _safe_get(props, "sku", "tier"),
        "network": _network_refs(props) or None,
    }


# Apply whitelisting to all resources
for key in rg_map:
    rg_map[key] = [whitelist_resource(r) for r in rg_map[key]]

print(f"Whitelisted properties for {sum(len(v) for v in rg_map.values())} resources across {len(rg_map)} RGs.")

Whitelisted properties for 15886 resources across 1482 RGs.


## Cell 6: Build Per-RG Type Matrix

In [49]:
# Build type counter and resource list per RG
rg_type_counts: dict[tuple[str, str], Counter] = {}
rg_resources: dict[tuple[str, str], list[dict]] = {}
singleton_keys: set[tuple[str, str]] = set()

for key, resources in rg_map.items():
    types = [r["type"] for r in resources if r.get("type")]
    counter = Counter(types)
    rg_type_counts[key] = counter
    rg_resources[key] = resources
    if len(set(types)) <= 1:
        singleton_keys.add(key)

multi_rg_keys = [k for k in rg_map if k not in singleton_keys]

# Tenant-wide type frequency
global_type_counter: Counter = Counter()
for c in rg_type_counts.values():
    global_type_counter.update(c)

print(f"Total RGs: {len(rg_map)}  |  Multi-type RGs: {len(multi_rg_keys)}  |  Singletons: {len(singleton_keys)}")
print("\nTop-10 resource types tenant-wide:")
for typ, cnt in global_type_counter.most_common(10):
    print(f"  {typ}: {cnt}")

# Build VOCAB (sorted list of all distinct types)
VOCAB = sorted(global_type_counter.keys())
print(f"\nVOCAB size: {len(VOCAB)}")

Total RGs: 1482  |  Multi-type RGs: 571  |  Singletons: 911

Top-10 resource types tenant-wide:
  microsoft.compute/virtualmachines/extensions: 3639
  microsoft.compute/galleries/images/versions: 1035
  microsoft.network/networksecuritygroups: 957
  microsoft.network/networkinterfaces: 809
  microsoft.compute/virtualmachines: 778
  microsoft.alertsmanagement/smartdetectoralertrules: 694
  microsoft.insights/metricalerts: 665
  microsoft.insights/actiongroups: 626
  microsoft.insights/components: 549
  microsoft.network/networkmanagers: 473

VOCAB size: 181


## Cell 7: BM25 Index

In [50]:
class BM25Index:
    """Okapi BM25 index over a corpus of token-lists (resource types per RG)."""

    def __init__(self, docs: list[list[str]], k1: float = 1.5, b: float = 0.75):
        self.k1 = k1
        self.b = b
        self.N = len(docs)
        self.docs = docs

        # Document lengths and average
        self.doc_lens = np.array([len(d) for d in docs], dtype=np.float64)
        self.avgdl = float(self.doc_lens.mean()) if self.N > 0 else 1.0

        # Term frequency per doc
        self.tf: list[Counter] = [Counter(d) for d in docs]

        # Document frequency and IDF per term
        df: Counter = Counter()
        for d in docs:
            df.update(set(d))
        self.idf: dict[str, float] = {}
        for term, n in df.items():
            self.idf[term] = math.log(1.0 + (self.N - n + 0.5) / (n + 0.5))

    def score_one(self, doc_idx: int, query_terms: list[str]) -> float:
        """BM25 score for a single document against the query."""
        score = 0.0
        dl = self.doc_lens[doc_idx]
        tf = self.tf[doc_idx]
        for q in query_terms:
            if q not in self.idf:
                continue
            freq = tf.get(q, 0)
            idf = self.idf[q]
            numerator = freq * (self.k1 + 1)
            denominator = freq + self.k1 * (1 - self.b + self.b * dl / self.avgdl)
            score += idf * numerator / denominator
        return score

    def score_all(self, query_terms: list[str]) -> np.ndarray:
        """BM25 scores for all documents against the query."""
        scores = np.array([self.score_one(i, query_terms) for i in range(self.N)])
        return scores


# Build corpus from multi-resource RGs only (singletons excluded)
corpus_keys = multi_rg_keys
corpus_docs: list[list[str]] = []
for key in corpus_keys:
    types = [r["type"] for r in rg_map[key] if r.get("type")]
    corpus_docs.append(types)

bm25 = BM25Index(corpus_docs, k1=BM25_K1, b=BM25_B)

print(f"BM25 corpus size: {bm25.N} documents (multi-type RGs)")
print(f"Average document length: {bm25.avgdl:.2f} types per RG")
print(f"Vocabulary size: {len(bm25.idf)} unique types in corpus")

BM25 corpus size: 571 documents (multi-type RGs)
Average document length: 24.65 types per RG
Vocabulary size: 160 unique types in corpus


## Cell 8: Must-Have Type Extraction

In [51]:
MUST_HAVE_SYSTEM_PROMPT = '''You are an Azure resource type classifier.

Given a user's infrastructure request, output ONLY a JSON object of the form:
  { "must_have_types": ["Microsoft.Web/sites", "Microsoft.Sql/servers"] }

Rules:
- Pick ONLY resource types that are absolutely required to fulfill the request.
- Do NOT include "nice to have" or supporting resources.
- Pick from the provided ALLOWED_TYPES list. If a type is not in that list, do not include it.
- 1-4 types is typical. Never more than 6.
- No explanations. No markdown. JSON only.
'''

must_have_user_msg = json.dumps({"request": USER_QUERY, "ALLOWED_TYPES": VOCAB})

print("Calling Azure OpenAI for must-have type extraction ...")
mh_response = aoai.chat.completions.create(
    model=AOAI_DEPLOYMENT,
    messages=[
        {"role": "system", "content": MUST_HAVE_SYSTEM_PROMPT},
        {"role": "user", "content": must_have_user_msg},
    ],
    # temperature=0.0,
    # max_tokens=256,
)

raw_mh = mh_response.choices[0].message.content.strip()
# Strip markdown fences if present
if raw_mh.startswith("```"):
    raw_mh = re.sub(r"^```(?:json)?\s*", "", raw_mh)
    raw_mh = re.sub(r"\s*```$", "", raw_mh)

mh_parsed = json.loads(raw_mh)
raw_types = mh_parsed.get("must_have_types", [])

# Intersect with VOCAB to drop hallucinated types
vocab_set = set(VOCAB)
must_have_types: list[str] = [t for t in raw_types if t in vocab_set]

dropped_types = set(raw_types) - vocab_set
if dropped_types:
    print(f"  Dropped hallucinated types: {dropped_types}")

print(f"Must-have types: {must_have_types}")

Calling Azure OpenAI for must-have type extraction ...
Must-have types: ['microsoft.web/sites', 'microsoft.web/serverfarms', 'microsoft.sql/servers', 'microsoft.sql/servers/databases']


## Cell 9: Hard Filter + BM25 Rank + Cutoff

In [52]:
must_have_set = set(must_have_types)

# Hard filter: each candidate RG must contain >= 1 must-have type
candidate_indices: list[int] = []
for i, key in enumerate(corpus_keys):
    rg_types = set(rg_type_counts[key].keys())
    if rg_types & must_have_set:
        candidate_indices.append(i)

print(f"Candidates after hard filter: {len(candidate_indices)} / {len(corpus_keys)} multi-type RGs")

if not candidate_indices:
    print("No matching RGs found. The insights bundle will use global fallback.")
    ranked = []
    confidence = "none"
else:
    # BM25 score against must-have list
    all_scores = bm25.score_all(must_have_types)
    candidate_scores = [(idx, all_scores[idx]) for idx in candidate_indices]
    candidate_scores.sort(key=lambda x: x[1], reverse=True)

    top_score = candidate_scores[0][1]
    cutoff = RELATIVE_CUTOFF * top_score

    # Keep RGs above cutoff, capped at TOP_K
    ranked = [(idx, sc) for idx, sc in candidate_scores if sc >= cutoff][:TOP_K]

    # Confidence tier: normalise top score by sum of must-have IDFs
    idf_sum = sum(bm25.idf.get(t, 0.0) for t in must_have_types)
    norm_score = top_score / idf_sum if idf_sum > 0 else 0.0

    if norm_score >= CONF_STRONG:
        confidence = "strong"
    elif norm_score >= CONF_PARTIAL:
        confidence = "partial"
    else:
        confidence = "weak"

    print(f"\nTop score: {top_score:.4f}  |  IDF sum: {idf_sum:.4f}  |  Normalised: {norm_score:.4f}  |  Confidence: {confidence}")
    print(f"Ranked RGs (cutoff={cutoff:.4f}):")
    for idx, sc in ranked:
        key = corpus_keys[idx]
        types_summary = dict(rg_type_counts[key].most_common(5))
        print(f"  [{sc:.4f}] {key[1]} (sub={key[0][:8]}...)  types={types_summary}")

Candidates after hard filter: 146 / 571 multi-type RGs

Top score: 16.5215  |  IDF sum: 12.7728  |  Normalised: 1.2935  |  Confidence: strong
Ranked RGs (cutoff=4.9564):
  [16.5215] r2d-int-uksouth (sub=58762415...)  types={'microsoft.sql/servers': 1, 'microsoft.sql/servers/databases': 1}
  [15.5531] elkimtest (sub=0ba674a6...)  types={'microsoft.network/networksecurityperimeters': 1, 'microsoft.sql/servers': 1, 'microsoft.sql/servers/databases': 1, 'microsoft.storage/storageaccounts': 1}
  [15.1694] shannon-hicks-rg (sub=0ba674a6...)  types={'microsoft.network/networksecurityperimeters': 2, 'microsoft.sql/servers/databases': 2, 'microsoft.alertsmanagement/smartdetectoralertrules': 1, 'microsoft.insights/components': 1, 'microsoft.keyvault/vaults': 1}
  [9.9870] kenieva-rg-test (sub=0ba674a6...)  types={'microsoft.keyvault/vaults': 4, 'microsoft.network/publicipaddresses': 4, 'microsoft.alertsmanagement/smartdetectoralertrules': 3, 'microsoft.compute/disks': 3, 'microsoft.insights/comp

## Cell 10: Conditional Aggregations

In [53]:
aggregations: list[dict] = []

if ranked:
    matched_keys = [corpus_keys[idx] for idx, _ in ranked]
    matched_resources = [r for k in matched_keys for r in rg_resources[k]]
    n_matched = len(matched_keys)

    # 1. Type presence: fraction of matched RGs containing each type
    type_presence: Counter = Counter()
    for k in matched_keys:
        for t in set(rg_type_counts[k].keys()):
            type_presence[t] += 1

    if n_matched >= MIN_AGG_N:
        for typ, count in type_presence.most_common():
            freq = count / n_matched
            aggregations.append({
                "claim": f"{typ} present in {freq:.0%} of matched RGs",
                "kind": "type_presence",
                "type": typ,
                "n": count,
                "of": n_matched,
                "frequency": round(freq, 3),
            })

    # 2. Region distribution: most common region across matched resources
    region_counter: Counter = Counter(r.get("location") for r in matched_resources if r.get("location"))
    if region_counter:
        top_region, top_region_count = region_counter.most_common(1)[0]
        total_locs = sum(region_counter.values())
        freq = top_region_count / total_locs
        aggregations.append({
            "claim": f"Most common region is '{top_region}' ({freq:.0%} of resources)",
            "kind": "region_distribution",
            "region": top_region,
            "n": top_region_count,
            "of": total_locs,
            "frequency": round(freq, 3),
        })

    # 3. SKU per type: most common SKU name for each type with enough instances
    sku_by_type: dict[str, list[str]] = defaultdict(list)
    for r in matched_resources:
        if r.get("sku") and isinstance(r["sku"], dict) and r["sku"].get("name"):
            sku_by_type[r["type"]].append(r["sku"]["name"])

    for typ, sku_names in sku_by_type.items():
        if len(sku_names) >= MIN_AGG_N:
            sku_counter = Counter(sku_names)
            top_sku, top_sku_count = sku_counter.most_common(1)[0]
            freq = top_sku_count / len(sku_names)
            aggregations.append({
                "claim": f"Most common SKU for {typ} is '{top_sku}' ({freq:.0%})",
                "kind": "sku_per_type",
                "type": typ,
                "sku": top_sku,
                "n": top_sku_count,
                "of": len(sku_names),
                "frequency": round(freq, 3),
            })

    # 4. Networking flags: VNet integration %, private endpoint %, managed identity %
    vnet_count = sum(1 for r in matched_resources if r.get("network") and r["network"].get("subnetId"))
    pe_count = sum(1 for r in matched_resources if r.get("network") and r["network"].get("hasPrivateEndpoints"))
    mi_count = sum(1 for r in matched_resources if r.get("identityType"))
    total_r = len(matched_resources)

    for label, count, kind in [
        ("VNet-integrated", vnet_count, "vnet_integration"),
        ("Private-endpoint-enabled", pe_count, "private_endpoints"),
        ("Managed-identity-enabled", mi_count, "managed_identity"),
    ]:
        freq = count / total_r if total_r else 0
        aggregations.append({
            "claim": f"{label}: {count}/{total_r} resources ({freq:.0%})",
            "kind": kind,
            "n": count,
            "of": total_r,
            "frequency": round(freq, 3),
        })

print(f"Generated {len(aggregations)} aggregation claims.")
for a in aggregations[:10]:
    print(f"  - {a['claim']}")
if len(aggregations) > 10:
    print(f"  ... and {len(aggregations) - 10} more.")

Generated 42 aggregation claims.
  - microsoft.sql/servers present in 80% of matched RGs
  - microsoft.sql/servers/databases present in 80% of matched RGs
  - microsoft.network/networksecurityperimeters present in 60% of matched RGs
  - microsoft.alertsmanagement/smartdetectoralertrules present in 60% of matched RGs
  - microsoft.keyvault/vaults present in 60% of matched RGs
  - microsoft.insights/components present in 60% of matched RGs
  - microsoft.operationalinsights/workspaces present in 40% of matched RGs
  - microsoft.web/sites present in 40% of matched RGs
  - microsoft.insights/actiongroups present in 40% of matched RGs
  - microsoft.storage/storageaccounts present in 20% of matched RGs
  ... and 32 more.


## Cell 11: Reference Exemplar

In [54]:
if ranked:
    top_idx, top_sc = ranked[0]
    top_key = corpus_keys[top_idx]
    exemplar = {
        "subscriptionId": top_key[0],
        "resourceGroup": top_key[1],
        "bm25_score": round(top_sc, 4),
        "type_counts": dict(rg_type_counts[top_key]),
        "resources": rg_resources[top_key],
    }
    print(f"Reference exemplar: {top_key[1]} (sub={top_key[0][:8]}..., score={top_sc:.4f})")
    print(f"  Types: {dict(rg_type_counts[top_key].most_common(8))}")
else:
    exemplar = None
    print("No exemplar available (no ranked matches).")

Reference exemplar: r2d-int-uksouth (sub=58762415..., score=16.5215)
  Types: {'microsoft.sql/servers': 1, 'microsoft.sql/servers/databases': 1}


## Cell 12: Assemble Insights Bundle

In [55]:
bundle = {
    "query": USER_QUERY,
    "required_types": must_have_types,
    "matched_rg_count": len(ranked),
    "confidence": confidence,
    "customer_conventions": aggregations,
    "reference_exemplar": exemplar,
}

# If no matches, add global fallback aggregations
if not ranked:
    top_types_global = [t for t, _ in global_type_counter.most_common(10)]
    region_counter_global: Counter = Counter(
        r.get("location")
        for resources in rg_map.values()
        for r in resources
        if r.get("location")
    )
    top_region_global = region_counter_global.most_common(1)[0][0] if region_counter_global else "unknown"

    bundle["global_fallback_aggregations"] = {
        "top_types": top_types_global,
        "top_region": top_region_global,
        "tenant_rg_count": len(rg_map),
        "tenant_resource_count": sum(len(v) for v in rg_map.values()),
    }
    print("No matched RGs — attached global fallback aggregations.")

print("\n=== Insights Bundle ===")
print(json.dumps({k: v for k, v in bundle.items() if k != "reference_exemplar"}, indent=2, default=str))
if exemplar:
    print(f"\n(reference_exemplar: {exemplar['resourceGroup']} with {len(exemplar['resources'])} resources)")


=== Insights Bundle ===
{
  "query": "Internal web app for the finance team with a relational database backend",
  "required_types": [
    "microsoft.web/sites",
    "microsoft.web/serverfarms",
    "microsoft.sql/servers",
    "microsoft.sql/servers/databases"
  ],
  "matched_rg_count": 5,
  "confidence": "strong",
  "customer_conventions": [
    {
      "claim": "microsoft.sql/servers present in 80% of matched RGs",
      "kind": "type_presence",
      "type": "microsoft.sql/servers",
      "n": 4,
      "of": 5,
      "frequency": 0.8
    },
    {
      "claim": "microsoft.sql/servers/databases present in 80% of matched RGs",
      "kind": "type_presence",
      "type": "microsoft.sql/servers/databases",
      "n": 4,
      "of": 5,
      "frequency": 0.8
    },
    {
      "claim": "microsoft.network/networksecurityperimeters present in 60% of matched RGs",
      "kind": "type_presence",
      "type": "microsoft.network/networksecurityperimeters",
      "n": 3,
      "of": 5,
    

## Cell 13: Generate Narrative Insights

In [56]:
INSIGHTS_PROMPT = """# Role and Objective
You are an expert Azure Insight Agent. Your mission is to analyze the user's existing infrastructure data and produce insights that inform downstream infrastructure plan generation.

# Process
1. Analyze the data and derive insights from dominant patterns in the user's existing infrastructure.
2. Re-examine the insights you produced. Check them for completeness and accuracy, and improve any that fall short.

# Insight Guidelines
When selecting resource properties to base insights on:
- Only consider properties that represent explicit user decisions affecting design.
- Never include properties involving runtime, versions, implementation details, app settings, default values, operational settings, or boilerplate configurations.
- Never include instance-specific properties of a resource.

### Structure of an Insight

Each insight must contain three parts: an observed pattern, the reasoning behind it, and a planning implication.
- The reasoning must be grounded in factual information from the data. Do not make assumptions.
- The planning implication must state concrete actions or decisions for infra planning that align with the user's requirements.
- The reasoning must clearly connect the observed pattern to the planning implication.

### Filtering

Use the following areas as a guide when deciding which resource properties are meaningful:
- Region
- Resource pairing
- Security posture
- Cost
- Naming and tagging conventions
- Azure policies

# Output

Return a JSON object with an "insights" key containing an array of insight strings.

```json
{
  "insights": [
    "Insight 1",
    "Insight 2",
    "Insight 3"
  ]
}
```

Each insight must be a single sentence with this structure: "[observed pattern]: [reasoning] [planning implication]"."""


bundle_json = json.dumps(bundle, indent=2, default=str)

print("Calling Azure OpenAI for narrative insight generation ...")
insight_response = aoai.chat.completions.create(
    model=AOAI_DEPLOYMENT,
    messages=[
        {"role": "system", "content": INSIGHTS_PROMPT},
        {"role": "user", "content": bundle_json},
    ],
    # temperature=0.3,
    # max_tokens=2048,
)

raw_insights = insight_response.choices[0].message.content.strip()

# Strip markdown fences if present
if raw_insights.startswith("```"):
    raw_insights = re.sub(r"^```(?:json)?\s*", "", raw_insights)
    raw_insights = re.sub(r"\s*```$", "", raw_insights)

parsed_insights = json.loads(raw_insights)

# Normalise: accept {"insights": [...]} or bare list
if isinstance(parsed_insights, dict) and "insights" in parsed_insights:
    insights_list: list[str] = parsed_insights["insights"]
elif isinstance(parsed_insights, list):
    insights_list = parsed_insights
else:
    raise ValueError(f"Unexpected insights format: {type(parsed_insights)}")

print(f"\nGenerated {len(insights_list)} insights:\n")
for i, insight in enumerate(insights_list, 1):
    print(f"  {i}. {insight}")

Calling Azure OpenAI for narrative insight generation ...

Generated 8 insights:

  1. Azure SQL dominance: 4 of 5 matched resource groups (80%) include both microsoft.sql/servers and microsoft.sql/servers/databases, indicating Azure SQL is the preferred relational backend; plan to provision an Azure SQL logical server and database as the default relational component for the finance web app.
  2. Hyperscale tendency for databases: the most common database SKU observed is HS_Gen5 (40% of database instances), suggesting existing databases favor Hyperscale Gen5 configurations; plan to evaluate Hyperscale (Gen5) vCore sizing and cost trade-offs when selecting the database tier for the finance workload.
  3. Key Vault adoption: microsoft.keyvault/vaults appear in 60% of matched resource groups, showing a pattern of using Key Vault for secrets and certificates; plan to integrate Azure Key Vault for DB credentials and TLS artifacts and design secure access (preferably via managed identities) 

## Cell 14: Save Output

In [57]:
output_path = Path("results/bm25.json")
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(insights_list, f, indent=2)

print(f"Insights saved to {output_path}")
print(f"\n=== Final Insights ({len(insights_list)} total) ===")
for i, insight in enumerate(insights_list, 1):
    print(f"  {i}. {insight}")

Insights saved to .azure\insights_bm25.json

=== Final Insights (8 total) ===
  1. Azure SQL dominance: 4 of 5 matched resource groups (80%) include both microsoft.sql/servers and microsoft.sql/servers/databases, indicating Azure SQL is the preferred relational backend; plan to provision an Azure SQL logical server and database as the default relational component for the finance web app.
  2. Hyperscale tendency for databases: the most common database SKU observed is HS_Gen5 (40% of database instances), suggesting existing databases favor Hyperscale Gen5 configurations; plan to evaluate Hyperscale (Gen5) vCore sizing and cost trade-offs when selecting the database tier for the finance workload.
  3. Key Vault adoption: microsoft.keyvault/vaults appear in 60% of matched resource groups, showing a pattern of using Key Vault for secrets and certificates; plan to integrate Azure Key Vault for DB credentials and TLS artifacts and design secure access (preferably via managed identities) rath